# B7: one decoder layer and residual connections

**Goal:** connect B2-B6 into one flattened-token Llama decoder layer.

```
attention_delta = attention(norm1(x))
after_attention = x + attention_delta
mlp_delta = mlp(norm2(after_attention))
output = after_attention + mlp_delta
```

Keep the draft, add an attention edit, then add an MLP edit. A residual addition keeps the unnormalized draft; normalization happens only on the branch input.

**Spec:** hidden states `(N, hidden)` and `AttentionMetadata(positions, sequence_ids)` -> `(N, hidden)`. Two RMSNorms, one attention block, one MLP, two residual additions. HF-compatible weight names. No cache or full model yet.

Run top to bottom in **Colab on a T4**.

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


In [ ]:
import importlib
import torch
import tinyserve.models.attention, tinyserve.models.llama, tinyserve.probe
for module in (tinyserve.models.attention, tinyserve.models.llama, tinyserve.probe):
    importlib.reload(module)
from tinyserve.models.attention import AttentionMetadata
from tinyserve.models.llama import DecoderLayer
from tinyserve.envinfo import collect_env, save_result

if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
torch.backends.cuda.matmul.allow_tf32 = False
torch.set_float32_matmul_precision("highest")
print(collect_env())


## 1. Test first

Run B4-B7 checks together. New checks cover residual/norm order, identity with zero output branches, future-token causality, packed isolation, weight keys and HF layer parity under MHA/GQA/MQA.

Failures stop the notebook. Read skips: HF and CUDA checks should be available here. Small FP16-vs-FP32 tests use `rtol=atol=8e-3` for accumulated intermediate rounding; real HF checks use FP32 at `1e-5`.

In [ ]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_b4_attention.py", "tests/test_b5_gqa.py",
                           "tests/test_b6_mlp.py", "tests/test_b7_decoder.py", "-v", "-ra"], capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


## 2. Residuals by hand

Suppose the stream is `[2, 3]`, attention contributes `[0.1, -0.2]`, and the MLP contributes `[0.4, 0.5]` on its updated input:

```
after attention: [2, 3] + [0.1, -0.2] = [2.1, 2.8]
after MLP:       [2.1, 2.8] + [0.4, 0.5] = [2.5, 3.3]
```

These deltas are illustrative, not outputs of trained weights. If both branches produce zero, the complete layer must return the original input exactly.

In [ ]:
from transformers import LlamaConfig
toy_config = LlamaConfig(hidden_size=64, intermediate_size=128, num_attention_heads=4,
                         num_key_value_heads=2, head_dim=16, num_hidden_layers=2, vocab_size=32)
torch.manual_seed(7)
identity_layer = DecoderLayer(toy_config, device=DEVICE, dtype=torch.float32).eval()
demo_input = torch.randn(5, 64, device=DEVICE)
demo_metadata = AttentionMetadata(torch.arange(5, device=DEVICE))
with torch.no_grad():
    identity_layer.self_attn.o_proj.weight.zero_()
    identity_layer.mlp.down_proj.weight.zero_()
    identity_output = identity_layer(demo_input, demo_metadata)
identity_max_diff = (identity_output - demo_input).abs().max().item()
print("zero-branch output vs input:", identity_max_diff)
assert identity_max_diff == 0.0


## 3. Check the two branch edits explicitly

The second norm reads **after_attention**, not the original input. Neither residual stream is replaced by its normalized version. Inspect the shapes: the branches return the same width as the stream.

In [ ]:
demo_layer = DecoderLayer(toy_config, device=DEVICE, dtype=torch.float32).eval()
with torch.no_grad():
    attention_delta = demo_layer.self_attn(demo_layer.input_layernorm(demo_input),
                                          demo_metadata.positions, demo_metadata.allowed_mask())
    after_attention = demo_input + attention_delta
    mlp_delta = demo_layer.mlp(demo_layer.post_attention_layernorm(after_attention))
    explicit_output = after_attention + mlp_delta
    demo_output = demo_layer(demo_input, demo_metadata)
wiring_max_diff = (explicit_output - demo_output).abs().max().item()
print("stream:", tuple(demo_input.shape), "attention edit:", tuple(attention_delta.shape), "MLP edit:", tuple(mlp_delta.shape))
print("explicit steps vs layer:", wiring_max_diff)
assert wiring_max_diff == 0.0


## 4. Whole-layer causality and packed isolation

Change the last two tokens: the first three outputs must stay the same. Then pack a 3-token request and a 2-token request, restart positions, and assign distinct IDs. Packed outputs must match running the requests separately.

The metadata object describes token layout. It is not a KV cache or a scheduler.

In [ ]:
changed_input = demo_input.clone()
changed_input[3:] += 10
packed_metadata = AttentionMetadata(torch.tensor([0, 1, 2, 0, 1], device=DEVICE),
                                    torch.tensor([0, 0, 0, 1, 1], device=DEVICE))
with torch.no_grad():
    changed_output = demo_layer(changed_input, demo_metadata)
    packed_output = demo_layer(demo_input, packed_metadata)
    separate_output = torch.cat([
        demo_layer(demo_input[:3], AttentionMetadata(torch.arange(3, device=DEVICE))),
        demo_layer(demo_input[3:], AttentionMetadata(torch.arange(2, device=DEVICE))),
    ])
prefix_max_diff = (demo_output[:3] - changed_output[:3]).abs().max().item()
packed_max_diff = (packed_output - separate_output).abs().max().item()
print("earlier-token difference:", prefix_max_diff, "packed vs separate difference:", packed_max_diff)
assert prefix_max_diff == 0.0
torch.testing.assert_close(packed_output, separate_output, rtol=1e-5, atol=1e-6)


## 5. All 22 isolated real HF layers

Load TinyLlama in FP32 with eager attention. Capture each HF decoder's real input/output, load that layer's weights into our decoder, and compare. The probe asserts `rtol=atol=1e-5`.

**Important:** each comparison starts from HF's input to that layer. B8 will test our own full stack, where one layer's output feeds the next, then check final logits.

FP32 model weights take about 4.4 GB. Close other GPU-heavy kernels if memory is low.

In [ ]:
from tinyserve.basics import load_model, encode
from tinyserve.probe import compare_decoder_layers
tokenizer, model = load_model(device=DEVICE, dtype=torch.float32)
model.set_attn_implementation("eager")
prompt_ids = encode(tokenizer, "The capital of France is", DEVICE)
decoder_rows = compare_decoder_layers(model, prompt_ids)
assert len(decoder_rows) == model.config.num_hidden_layers == 22
for row in decoder_rows:
    print(f"{row['name']}: max |diff| {row['max_abs_diff']:.2e}, shape {row['shape']}, output max {row['output_max_abs']:.3f}")


## 6. Save evidence and three interview questions

1. **What is a residual?** Add a branch's output to the existing stream instead of replacing it.
2. **What is pre-norm?** Normalize the input to each branch before computation, while keeping the residual stream unnormalized.
3. **Does isolated parity prove full-model parity?** No. B8 checks accumulated layer outputs, final norm and logits.

Save with outputs. The min/max range below is across layers, not repeated performance trials. After validation, retain results and update the guide/tracker.

**Next: B8 full model, weight loading and FP32 logits/token parity.**

In [ ]:
layer_differences = [row["max_abs_diff"] for row in decoder_rows]
saved_path = save_result("B7_decoder_layer", {
    "hidden_size": 2048, "layers_compared": len(decoder_rows),
    "zero_branch_identity_max_abs_diff": identity_max_diff,
    "explicit_wiring_max_abs_diff": wiring_max_diff,
    "prefix_max_abs_diff": prefix_max_diff,
    "packed_max_abs_diff": packed_max_diff,
    "real_model_dtype": "float32", "real_model_backend": "eager",
    "real_model_comparison": decoder_rows,
    "real_model_diff_range_across_layers": {"min": min(layer_differences), "max": max(layer_differences)},
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory still held by this kernel: {torch.cuda.memory_allocated() / 2**20:.0f} MiB")
